<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/generative/diffusion-parameterizations/diffusion-parameterizations.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("diffusion-parameterizations", ...)`


# Diffusion Schedules and Parameterizations: ε, x₀, v, SNR

**Difficulty**: 🟡 Medium

---

### Problem Statement

Every Gaussian diffusion model (DDPM, Stable Diffusion, Imagen, video models) shares one forward process, $x_t = \alpha_t x_0 + \sigma_t \varepsilon$. They differ in three things. The **schedule** fixes how fast the signal-to-noise ratio decays with $t$. The **parameterization** decides whether the network predicts $\varepsilon$, $x_0$, $v$ or the score. The **loss weighting** decides how much each noise level counts. Any one prediction can be converted into the others, and a diffusion codebase is mostly this bookkeeping: diffusers' schedulers, `guided-diffusion`'s `GaussianDiffusion`, k-diffusion's wrappers. It is also where the silent bugs live: an off-by-one schedule array, a flipped $v$ sign, $\alpha_t$ confused with $\bar\alpha_t$, or a v-prediction model trained with the ε loss weights.

You will implement these pieces as small, vectorized, exact functions and check them against independent float64 references. This builds on `v3/modern-architectures/ddpm` (forward process and ancestral sampler) and `v3/modern-architectures/ddim-cfg` (DDIM's $\hat x_0$), and connects to `v4/generative/flow-matching`.

**Notation and conventions (graded)**

| symbol | meaning | in code |
|---|---|---|
| $t = 1, \dots, T$ | paper timestep | 0-based array index $i = t - 1$ |
| $\beta_t$ | per-step noise variance | `betas[i]` |
| $\bar\alpha_t = \prod_{s \le t}(1 - \beta_s)$, with $\bar\alpha_0 := 1$ | cumulative signal fraction | `alpha_bar[i]` |
| $\alpha_t := \sqrt{\bar\alpha_t}$, $\;\sigma_t := \sqrt{1 - \bar\alpha_t}$ | signal and noise **scales** (VDM notation, Kingma et al. 2021), with $\alpha_t^2 + \sigma_t^2 = 1$ | |
| $x_t = \alpha_t x_0 + \sigma_t\varepsilon$, $\;\varepsilon \sim \mathcal N(0, I)$ | the marginal $q(x_t \mid x_0)$ | |
| $\mathrm{SNR}(t) = \alpha_t^2 / \sigma_t^2 = \bar\alpha_t / (1 - \bar\alpha_t)$ | signal-to-noise ratio | |

⚠️ DDPM (Ho et al. 2020) uses $\alpha_t$ for the **per-step** factor $1 - \beta_t$. In this notebook $\alpha_t$ always means $\sqrt{\bar\alpha_t}$, and the per-step factor is always written $1 - \beta_t$.

Schedules are computed in **float64**, as in Ho et al.'s and Nichol & Dhariwal's reference code.

---

### Requirements

**Schedules** (each returns a `(T,)` float64 tensor)

1. **`linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02)`**: $\beta_1 = $ `beta_start`, …, $\beta_T = $ `beta_end`, equally spaced (Ho et al. 2020, with $T = 1000$).
2. **`cosine_alpha_bar(T, s=0.008, max_beta=0.999)`**: $\bar\alpha_1, \dots, \bar\alpha_T$ of the cosine schedule (Nichol & Dhariwal 2021, eq. 17), with the clipping the paper describes right after it:
$$f(t) = \cos^2\!\Big(\frac{t/T + s}{1 + s}\cdot\frac{\pi}{2}\Big), \qquad \beta_t = \min\!\Big(1 - \frac{f(t)}{f(t-1)},\ \texttt{max\_beta}\Big), \qquad \bar\alpha_t = \prod_{s \le t} (1 - \beta_s).$$
   Without clipping this is just $\bar\alpha_t = f(t)/f(0)$. Clipping changes only the steps whose $\beta_t$ exceeds `max_beta`; with the defaults that is only the last one, whose unclipped $\beta_T$ is 1.

**Conversions.** `alpha_bar` is a `(B,)` tensor, one $\bar\alpha$ per sample, to be broadcast over the trailing dimensions of the other arguments (any shape `(B, ...)`).

3. **`eps_to_x0(x_t, eps, alpha_bar)`** → $x_0 = (x_t - \sigma_t\varepsilon)/\alpha_t$.
4. **`x0_to_eps(x_t, x0, alpha_bar)`** → $\varepsilon = (x_t - \alpha_t x_0)/\sigma_t$.
5. **`v_from_x0_eps(x0, eps, alpha_bar)`** → $v = \alpha_t\varepsilon - \sigma_t x_0$ (Salimans & Ho 2022).
6. **`v_to_x0_eps(x_t, v, alpha_bar)`** → the tuple `(x0, eps)` recovered from $x_t$ and $v$. It must not divide, so it is valid even at $\bar\alpha = 0$ and $\bar\alpha = 1$.
7. **`eps_to_score(eps, alpha_bar)`** → $\nabla_{x_t}\log q(x_t \mid x_0)$ expressed through $\varepsilon$.

**SNR and loss weights**

8. **`snr(alpha_bar)`** → $\bar\alpha / (1 - \bar\alpha)$, elementwise, for any shape: $0$ at $\bar\alpha = 0$ and `inf` at $\bar\alpha = 1$.
9. **`min_snr_weight(alpha_bar, gamma=5.0, target="eps")`** → elementwise loss weights for **Min-SNR-γ** (Hang et al., ICCV 2023), for a model trained with the MSE on `target` ∈ {`"eps"`, `"x0"`, `"v"`}. Min-SNR-γ weights the **$x_0$ error** by $\min(\mathrm{SNR}, \gamma)$. The weight for each target must make the loss equal for the same prediction:
$$w_{\text{target}}\cdot\lVert \text{target} - \widehat{\text{target}}\rVert^2 = \min(\mathrm{SNR}, \gamma)\cdot\lVert x_0 - \hat x_0\rVert^2,$$
   where $\hat x_0$, $\hat\varepsilon$, $\hat v$ are conversions of one another at the same $x_t$. Derive the three formulas. At $\mathrm{SNR} = 0$, return the limit $\mathrm{SNR}\to 0^+$ (no NaN).
10. **`rescale_zero_terminal_snr(betas)`** → new `(T,)` betas following Lin et al. (*Common Diffusion Noise Schedules and Sample Steps are Flawed*, WACV 2024, arXiv:2305.08891), Algorithm 1. With $s_t = \sqrt{\bar\alpha_t}$:
$$s_t \leftarrow (s_t - s_T)\,\frac{s_1}{s_1 - s_T}, \qquad \bar\alpha_t = s_t^2, \qquad \beta_1 = 1 - \bar\alpha_1, \qquad \beta_t = 1 - \bar\alpha_t / \bar\alpha_{t-1}\ \ (t \ge 2).$$
   Afterwards $\bar\alpha_T = 0$ exactly (so $\beta_T = 1$ and the terminal SNR is 0), and $\bar\alpha_1$ is unchanged. Do not modify the input in place.
11. **`q_posterior(x0, x_t, t, betas)`** → `(mean, var)` of $q(x_{t-1}\mid x_t, x_0) = \mathcal N(\tilde\mu_t, \tilde\beta_t I)$ (Ho et al. 2020, eqs. 6–7):
$$\tilde\mu_t = \frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1 - \bar\alpha_t}\,x_0 + \frac{\sqrt{1 - \beta_t}\,(1 - \bar\alpha_{t-1})}{1 - \bar\alpha_t}\,x_t, \qquad \tilde\beta_t = \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}\,\beta_t.$$
   `t` is a `(B,)` long tensor of **0-based** indices (paper step = index + 1), one per sample. `x0` and `x_t` have shape `(B, ...)`; `mean` has the shape of `x_t` and `var` has shape `(B,)`. At index 0, $\bar\alpha_0 = 1$ gives `mean = x0`, `var = 0`.

---

### Constraints

- ✅ Plain `torch` ops: `cumprod`, `clamp`, `cat`, indexing with a tensor of timesteps.
- ❌ No Python loops over timesteps or samples (the reference oracles in the tests use loops; your functions should not).
- ❌ No `diffusers`, `k-diffusion` or other diffusion libraries.
- ❌ Don't clamp away the singularities of `eps_to_x0`, `x0_to_eps` and `eps_to_score` at $\bar\alpha \in \{0, 1\}$. They are real, and they are why v-prediction exists.

---

<details>
  <summary>💡 Hint</summary>

  - Because $\alpha^2 + \sigma^2 = 1$, you can write $\alpha = \cos\phi$ and $\sigma = \sin\phi$. Then $(x_t, v)$ is a **rotation** of $(x_0, \varepsilon)$ by the angle $\phi$. What is the inverse of a rotation?
  - A `(B,)` tensor times a `(B, C, H, W)` tensor broadcasts against the **last** axis. That is wrong here, and when `B == W` it fails silently.
  - For the cosine schedule, evaluate $f$ at $t = 0, 1, \dots, T$ (that is $T + 1$ points) and divide neighbours.
  - Min-SNR: for a fixed $x_t$, how does an error in $\hat\varepsilon$ show up as an error in $\hat x_0$? And an error in $\hat v$?
  - Posterior: what is $\bar\alpha_{t-1}$ when $t$ is index 0? (Python's `abar[-1]` is not it.)

</details>

---

In [ ]:
import math

import torch

F64 = torch.float64
torch.manual_seed(0)

In [ ]:
def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    """
    Returns:
        betas: (T,) float64, betas[i] = beta_{i+1}, linearly spaced from beta_start to beta_end.
    """
    # TODO
    raise NotImplementedError


def cosine_alpha_bar(T, s=0.008, max_beta=0.999):
    """
    Returns:
        alpha_bar: (T,) float64, alpha_bar[i] = abar_{i+1} of the cosine schedule, built from
                   betas clipped at max_beta (Nichol & Dhariwal 2021, eq. 17).
    """
    # TODO
    raise NotImplementedError

In [ ]:
def eps_to_x0(x_t, eps, alpha_bar):
    """x_t, eps: (B, ...); alpha_bar: (B,). Returns x0 = (x_t - sigma * eps) / alpha, shape (B, ...)."""
    # TODO
    raise NotImplementedError


def x0_to_eps(x_t, x0, alpha_bar):
    """x_t, x0: (B, ...); alpha_bar: (B,). Returns eps = (x_t - alpha * x0) / sigma, shape (B, ...)."""
    # TODO
    raise NotImplementedError


def v_from_x0_eps(x0, eps, alpha_bar):
    """x0, eps: (B, ...); alpha_bar: (B,). Returns v = alpha * eps - sigma * x0, shape (B, ...)."""
    # TODO
    raise NotImplementedError


def v_to_x0_eps(x_t, v, alpha_bar):
    """x_t, v: (B, ...); alpha_bar: (B,). Returns the tuple (x0, eps), each (B, ...), with no division."""
    # TODO
    raise NotImplementedError


def eps_to_score(eps, alpha_bar):
    """eps: (B, ...); alpha_bar: (B,). Returns grad_{x_t} log q(x_t | x0) in terms of eps, shape (B, ...)."""
    # TODO
    raise NotImplementedError

In [ ]:
def snr(alpha_bar):
    """alpha_bar: any shape. Returns abar / (1 - abar), same shape."""
    # TODO
    raise NotImplementedError


def min_snr_weight(alpha_bar, gamma=5.0, target="eps"):
    """
    Args:
        alpha_bar: any shape.   gamma: the Min-SNR clip.   target: "eps", "x0" or "v".
    Returns:
        Per-element weight for the MSE on `target` such that it equals min(SNR, gamma) times the
        x0 MSE of the same prediction; finite at SNR = 0 (the SNR -> 0+ limit).
    """
    # TODO
    raise NotImplementedError

In [ ]:
def rescale_zero_terminal_snr(betas):
    """
    Args:
        betas: (T,) float64 (not modified).
    Returns:
        (T,) float64 betas whose sqrt(abar) is shifted to 0 at t = T and rescaled to keep its
        t = 1 value (Lin et al. 2024, Algorithm 1). beta_T = 1, abar_T = 0 exactly.
    """
    # TODO
    raise NotImplementedError


def q_posterior(x0, x_t, t, betas):
    """
    Args:
        x0, x_t: (B, ...).   t: (B,) long, 0-based indices.   betas: (T,) float64.
    Returns:
        mean: (B, ...), the mean of q(x_{t-1} | x_t, x0).
        var:  (B,), its variance beta~_t (0 at index 0).
    """
    # TODO
    raise NotImplementedError

In [ ]:
# Test 1: schedules
betas = linear_beta_schedule(1000)
assert betas.shape == (1000,) and betas.dtype == F64, (betas.shape, betas.dtype)
assert torch.allclose(betas, torch.linspace(1e-4, 0.02, 1000, dtype=F64), rtol=1e-12, atol=0)
assert betas[0].item() == 1e-4 and abs(betas[-1].item() - 0.02) < 1e-15


def cosine_reference(T, s=0.008, max_beta=0.999):
    # improved-diffusion's betas_for_alpha_bar loop, then a running product
    f = lambda u: math.cos((u + s) / (1 + s) * math.pi / 2) ** 2
    ab, out = 1.0, []
    for i in range(T):
        ab *= 1 - min(1 - f((i + 1) / T) / f(i / T), max_beta)
        out.append(ab)
    return torch.tensor(out, dtype=F64)


for T in (1000, 50):
    ab = cosine_alpha_bar(T)
    assert ab.shape == (T,) and ab.dtype == F64, (ab.shape, ab.dtype)
    assert torch.allclose(ab, cosine_reference(T), rtol=1e-10, atol=0), f"T = {T}"
assert torch.allclose(cosine_alpha_bar(20, max_beta=0.5), cosine_reference(20, max_beta=0.5), rtol=1e-10, atol=0), \
    "clip every beta above max_beta, not just the last one"
ab = cosine_alpha_bar(1000)
assert ab[0] < 1 and bool((ab[1:] < ab[:-1]).all()), "index 0 is abar_1 < 1; abar must strictly decrease"
print(f"Test 1 passed: abar_T = {torch.cumprod(1 - betas, 0)[-1].item():.2e} (linear), "
      f"{ab[-1].item():.2e} (cosine, last beta clipped)")

In [ ]:
# Test 2: conversions (float64, one abar per sample; B == last dim on purpose)
g = torch.Generator().manual_seed(0)
x0 = torch.randn(5, 3, 4, 5, dtype=F64, generator=g)
eps = torch.randn(5, 3, 4, 5, dtype=F64, generator=g)
ab = torch.tensor([0.9999, 0.93, 0.5, 0.07, 3e-4], dtype=F64)
a, s = ab.sqrt().view(5, 1, 1, 1), (1 - ab).sqrt().view(5, 1, 1, 1)
x_t = a * x0 + s * eps

assert torch.allclose(eps_to_x0(x_t, eps, ab), x0) and torch.allclose(x0_to_eps(x_t, x0, ab), eps)
v = v_from_x0_eps(x0, eps, ab)
assert torch.allclose(v, a * eps - s * x0), "v = alpha * eps - sigma * x0"
x0_back, eps_back = v_to_x0_eps(x_t, v, ab)
assert torch.allclose(x0_back, x0) and torch.allclose(eps_back, eps), "v_to_x0_eps must invert"

# v is the velocity of x(phi) = cos(phi) x0 + sin(phi) eps along the angle phi = atan2(sigma, alpha)
phi = torch.atan2((1 - ab).sqrt(), ab.sqrt()).view(5, 1, 1, 1)
h = 1e-6
dx_dphi = ((torch.cos(phi + h) - torch.cos(phi - h)) * x0 + (torch.sin(phi + h) - torch.sin(phi - h)) * eps) / (2 * h)
assert torch.allclose(dx_dphi, v, atol=1e-6), "v must equal d x / d phi"

# the score of q(x_t | x0) = N(alpha x0, sigma^2 I), by autograd
xt_leaf = x_t.clone().requires_grad_(True)
logq = torch.distributions.Normal(a * x0, s).log_prob(xt_leaf).sum()
(score,) = torch.autograd.grad(logq, xt_leaf)
assert torch.allclose(eps_to_score(eps, ab), score), "score of q(x_t | x0) in terms of eps"

# v needs no division: it works at abar = 0 (pure noise) and abar = 1 (clean data)
x0e, ee = v_to_x0_eps(x_t[:2], v[:2], torch.tensor([0.0, 1.0], dtype=F64))
assert torch.allclose(x0e[0], -v[0]) and torch.allclose(ee[0], x_t[0]), "abar = 0: x0 = -v, eps = x_t"
assert torch.allclose(x0e[1], x_t[1]) and torch.allclose(ee[1], v[1]), "abar = 1: x0 = x_t, eps = v"
print("Test 2 passed: eps <-> x0, v <-> (x0, eps), v = dx/dphi, score")

In [ ]:
# Test 3: SNR and Min-SNR-gamma weights
grid = torch.tensor([0.9999, 0.95, 0.8, 0.5, 0.2, 0.01], dtype=F64)
assert torch.allclose(snr(grid), grid / (1 - grid))
edge = snr(torch.tensor([0.0, 1.0], dtype=F64))
assert edge[0].item() == 0.0 and math.isinf(edge[1].item()), "SNR is 0 at abar = 0 and inf at abar = 1"
assert torch.allclose(min_snr_weight(grid, 5.0, "x0"), snr(grid).clamp(max=5.0)), "x0 weight = min(SNR, gamma)"

# the defining property: one prediction error, measured on each target and weighted by that
# target's weight, gives the same loss (x_t, x0, eps, v, a, s, ab from Test 2)
eps_hat = eps + 0.3 * torch.randn(eps.shape, dtype=F64, generator=g)
x0_hat = (x_t - s * eps_hat) / a
v_hat = a * eps_hat - s * x0_hat
mse = lambda p, q: ((p - q) ** 2).flatten(1).mean(1)
loss_x0 = min_snr_weight(ab, 5.0, "x0") * mse(x0_hat, x0)
assert torch.allclose(min_snr_weight(ab, 5.0, "eps") * mse(eps_hat, eps), loss_x0), "eps weight"
assert torch.allclose(min_snr_weight(ab, 5.0, "v") * mse(v_hat, v), loss_x0), "v weight"
w0 = {k: min_snr_weight(torch.zeros(1, dtype=F64), 5.0, k).item() for k in ("eps", "x0", "v")}
assert w0 == {"eps": 1.0, "x0": 0.0, "v": 0.0}, f"weights at SNR = 0 must be the SNR -> 0 limits, got {w0}"
print("Test 3 passed: SNR; Min-SNR weights equalize the eps / x0 / v losses")

In [ ]:
# Test 4: zero terminal SNR (Lin et al. 2024, Algorithm 1)
def zero_snr_reference(betas):
    # Algorithm 1 as a plain Python loop
    prod, s = 1.0, []
    for b in betas.tolist():
        prod *= 1 - b
        s.append(math.sqrt(prod))
    s = [(x - s[-1]) * s[0] / (s[0] - s[-1]) for x in s]
    ab2 = [x * x for x in s]
    return torch.tensor([1 - ab2[0]] + [1 - ab2[i] / ab2[i - 1] for i in range(1, len(ab2))], dtype=F64)


sd_betas = torch.linspace(0.00085 ** 0.5, 0.012 ** 0.5, 1000, dtype=F64) ** 2   # Stable Diffusion's "scaled_linear"
keep = sd_betas.clone()
zt_betas = rescale_zero_terminal_snr(sd_betas)
assert torch.equal(sd_betas, keep), "do not modify the input"
ab_old, ab_new = torch.cumprod(1 - sd_betas, 0), torch.cumprod(1 - zt_betas, 0)
assert zt_betas[-1].item() == 1.0 and ab_new[-1].item() == 0.0, "terminal SNR must be exactly 0 (beta_T = 1)"
assert torch.allclose(ab_new[0], ab_old[0], rtol=1e-12, atol=0), "SNR at t = 1 must be preserved"
assert torch.allclose(zt_betas, zero_snr_reference(sd_betas), rtol=1e-9, atol=1e-12)
print(f"Test 4 passed: SD's terminal SNR {snr(ab_old[-1]).item():.6f} -> {snr(ab_new[-1]).item()}")

In [ ]:
# Test 5: q(x_{t-1} | x_t, x0) against Bayes' rule
betas50 = linear_beta_schedule(50, 1e-3, 0.2)
abar50 = torch.cumprod(1 - betas50, 0)
t = torch.tensor([0, 1, 7, 30, 49])
x0p = torch.randn(5, 2, 3, 5, dtype=F64, generator=g)
xtp = torch.randn(5, 2, 3, 5, dtype=F64, generator=g)
mean, var = q_posterior(x0p, xtp, t, betas50)
assert mean.shape == xtp.shape and var.shape == (5,), (mean.shape, var.shape)
assert torch.allclose(mean[0], x0p[0]) and abs(var[0].item()) < 1e-15, "index 0: point mass at x0"
for i in range(1, 5):
    # q(x_{t-1} | x0) = N(sqrt(abar_prev) x0, 1 - abar_prev) times q(x_t | x_{t-1}) = N(sqrt(1 - b) x_{t-1}, b)
    b, ab_prev = betas50[t[i]].item(), abar50[t[i] - 1].item()
    prec = 1 / (1 - ab_prev) + (1 - b) / b
    want = (math.sqrt(ab_prev) * x0p[i] / (1 - ab_prev) + math.sqrt(1 - b) * xtp[i] / b) / prec
    assert abs(var[i].item() - 1 / prec) < 1e-12 and torch.allclose(mean[i], want), f"index {t[i].item()}"
print("Test 5 passed: posterior mean and variance match Bayes' rule; index 0 is a point mass")

In [ ]:
# Four schedules in SNR terms. log-SNR = log(abar / (1 - abar)), the axis on which diffusion
# schedules are compared (Kingma et al. 2021).
schedules = {
    "linear (DDPM)": torch.cumprod(1 - linear_beta_schedule(1000), 0),
    "cosine (iDDPM)": cosine_alpha_bar(1000),
    "scaled-linear (SD)": torch.cumprod(1 - sd_betas, 0),
    "SD + zero terminal SNR": torch.cumprod(1 - zt_betas, 0),
}
print(f"{'log-SNR at t =':24s}{'1':>8s}{'250':>8s}{'500':>8s}{'750':>8s}{'1000':>8s}    SNR_T")
for name, abar in schedules.items():
    lam = torch.log(snr(abar))
    print(f"{name:24s}" + "".join(f"{lam[i].item():8.2f}" for i in (0, 249, 499, 749, 999))
          + f"    {snr(abar[-1]).item():.3g}")

# The weight each plain loss puts on the x0 error: eps-MSE = SNR * x0-MSE, v-MSE = (SNR + 1) * x0-MSE.
# Min-SNR-5 caps it at 5, so the many nearly clean (high-SNR) steps stop dominating.
abar = schedules["linear (DDPM)"]
print(f"\n{'t':>5s}{'SNR':>11s}{'eps-loss':>11s}{'v-loss':>11s}{'min-SNR-5':>11s}   (weight on the x0 error)")
for i in (0, 99, 299, 499, 999):
    r = snr(abar[i]).item()
    print(f"{i + 1:5d}{r:11.4g}{r:11.4g}{r + 1:11.4g}{min(r, 5.0):11.4g}")

# At zero terminal SNR, x_T is pure noise: even a perfect eps-prediction says nothing about x0
# (x0 = (x_T - sigma eps) / alpha = 0 / 0), while v = alpha eps - sigma x0 = -x0 remains a target.
ab_T = torch.zeros(1, dtype=F64)
x0_d, eps_d = torch.randn(1, 4, dtype=F64, generator=g), torch.randn(1, 4, dtype=F64, generator=g)
x_T = eps_d.clone()
print("\nx0 from a perfect eps-prediction at t = T:", eps_to_x0(x_T, eps_d, ab_T))
print("x0 from a perfect v-prediction at t = T:  ", v_to_x0_eps(x_T, v_from_x0_eps(x0_d, eps_d, ab_T), ab_T)[0])
print("true x0:                                  ", x0_d)

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("diffusion-parameterizations",
      linear_beta_schedule=linear_beta_schedule, cosine_alpha_bar=cosine_alpha_bar,
      eps_to_x0=eps_to_x0, x0_to_eps=x0_to_eps, v_from_x0_eps=v_from_x0_eps,
      v_to_x0_eps=v_to_x0_eps, eps_to_score=eps_to_score, snr=snr,
      min_snr_weight=min_snr_weight, rescale_zero_terminal_snr=rescale_zero_terminal_snr,
      q_posterior=q_posterior)